<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane: CTR / Engagement Opportunity Scoring

**Lane:** CTR / Engagement Opportunity Scoring  
**Goal:** learn whether a page is likely to remain below its current position-adjusted CTR benchmark in the following month, then rank pages for review.

This notebook deliberately keeps the Week-4 baseline frozen and compares the learned model to that baseline on the **same client-held-out test split**.

**Primary metric:** Precision@10.  
**Secondary metrics:** Precision@50, average precision, ROC-AUC, and the test-set base rate.

No future-window fields are used as model features. March is used only to construct the outcome label.


## 1. Method choice and why

I chose **Logistic Regression** because this lane is a small, interpretable ranking problem rather than a need for a highly complex predictor. The model estimates the probability that a page's next-month CTR will be below the position-tier benchmark established from the decision-month data.

Why this is appropriate:
- the Week-4 rule already uses CTR, impressions, and position;
- Logistic Regression gives an auditable probability score and signed feature coefficients;
- the feature count is intentionally small;
- the model is easy to compare against the frozen hand-written rule;
- complexity is not treated as success: the model only earns the capstone claim if it improves Precision@K on the held-out clients.

The split is **grouped by `client_hash_id`**, so pages from one client cannot appear in both train and test.


In [ ]:
# Setup: clone the user's repo if needed, install dependencies, and authenticate to the gated warehouse.
import os
import subprocess
import numpy as np
import pandas as pd

REPO_DIR = "/content/Flyrank"
REPO_URL = "https://github.com/Samarjamal326/Flyrank.git"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

subprocess.run(["pip", "-q", "install", "duckdb", "scikit-learn", "pandas", "numpy"], check=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face READ token in Colab Secrets "
        "with the name HF_TOKEN and enable notebook access."
    )

import duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

os.makedirs("work/outputs", exist_ok=True)

print("Repository:", os.getcwd())
print("Warehouse connected.")


Repository: /content/Flyrank
Warehouse connected.


### Data construction

The decision month is **February 2026** and the outcome month is **March 2026**, matching the Week-3 data contract.

For February, we aggregate measured GSC rows to one row per `client_hash_id × content_hash_id` and calculate:
- impressions,
- clicks,
- CTR,
- impression-weighted average position,
- position tier.

The model features are all knowable at the February decision moment.

The March outcome is only used after the split to construct `future_ctr`; it is never passed into the model.


In [ ]:
# Build the February decision frame.
feb_sql = f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS feb_position_sum
    FROM {FEB}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    feb_impressions,
    feb_clicks,
    CASE
        WHEN feb_impressions > 0 THEN 100.0 * feb_clicks / feb_impressions
        ELSE NULL
    END AS feb_ctr,
    CASE
        WHEN feb_impressions > 0 THEN feb_position_sum / feb_impressions
        ELSE NULL
    END AS feb_avg_position
FROM feb
WHERE feb_impressions >= 100
  AND feb_position_sum IS NOT NULL
  AND feb_impressions IS NOT NULL
"""

feb = con.execute(feb_sql).df()

def position_tier(x):
    if x <= 3:
        return "top_3"
    if x <= 10:
        return "page_1"
    if x <= 20:
        return "striking"
    if x <= 50:
        return "page_3_5"
    return "deep"

feb["position_tier"] = feb["feb_avg_position"].map(position_tier)

print(f"February decision rows: {len(feb):,}")
display(feb.head())


February decision rows: 80,322


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,position_tier
0,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.000000,6.391489,page_1
1,client_3ffa76342f366962,content_456ab2db28595187,100.0,2.0,2.000000,4.200000,page_1
2,client_3ffa76342f366962,content_5573434837db89c5,198.0,6.0,3.030303,7.318182,page_1
3,client_3ffa76342f366962,content_7b17975c58745266,102.0,5.0,4.901961,4.450980,page_1
4,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,6.0,3.370787,3.393258,page_1


In [ ]:
# Build the March outcome frame.
mar_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS mar_impressions,
    SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS mar_clicks
FROM {MAR}
GROUP BY 1, 2
"""

mar = con.execute(mar_sql).df()

mar["future_ctr"] = np.where(
    mar["mar_impressions"] > 0,
    100.0 * mar["mar_clicks"] / mar["mar_impressions"],
    np.nan,
)

# Keep only pages with a measurable March outcome.
mar = mar.loc[
    mar["mar_impressions"].fillna(0) >= 100,
    ["client_hash_id", "content_hash_id", "mar_impressions", "mar_clicks", "future_ctr"]
].copy()

data = feb.merge(
    mar,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(f"Matched February → March rows with measurable outcome: {len(data):,}")
display(data.head())


Matched February → March rows with measurable outcome: 73,481


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,position_tier,mar_impressions,mar_clicks,future_ctr
0,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.0,6.391489,page_1,255.0,0.0,0.0
1,client_e547b89c05043229,content_2e296120acb03e93,660.0,0.0,0.0,61.680303,deep,3146.0,0.0,0.0
2,client_e547b89c05043229,content_516b7c0e8eec0cef,1139.0,0.0,0.0,49.698859,page_3_5,3496.0,0.0,0.0
3,client_e547b89c05043229,content_38b6c1a9aa29f801,1063.0,0.0,0.0,54.554092,deep,2751.0,0.0,0.0
4,client_e547b89c05043229,content_2ffd36f2a70be7e3,586.0,0.0,0.0,42.027304,page_3_5,1285.0,0.0,0.0


## 2. Split design

The split is performed **before** deriving the position-tier benchmark and target.

- 80% of clients: training
- 20% of clients: held-out test
- `random_state=42`
- no client appears in both groups

This matters because multiple content items can belong to the same client. A random row split would let client-specific patterns leak across train and test.


In [ ]:
# Grouped client holdout split.
groups = data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(data, groups=groups))

train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")
print(f"Train clients: {train['client_hash_id'].nunique():,}")
print(f"Test clients:  {test['client_hash_id'].nunique():,}")
print("Client overlap:", len(set(train["client_hash_id"]) & set(test["client_hash_id"])))


Train rows: 54,785
Test rows:  18,696
Train clients: 26
Test clients:  7
Client overlap: 0


### Leakage-safe target

The February position-tier median CTR is estimated **from training clients only**.

A positive outcome label means:

> March future CTR is below the February benchmark for the page's February position tier.

This keeps the target aligned with the opportunity-scoring lane while avoiding `trend_direction`, `trend_pct`, or any other label-derived field.

The February benchmark is a training-only statistic. The same benchmark is then applied unchanged to the test clients for both the baseline and the model evaluation.


In [ ]:
# Fit the position-tier CTR benchmark on TRAINING clients only.
tier_benchmark = (
    train.groupby("position_tier", observed=True)["feb_ctr"]
    .median()
    .rename("tier_benchmark_ctr")
)

train = train.join(tier_benchmark, on="position_tier")
test = test.join(tier_benchmark, on="position_tier")

# Positive = next-month CTR is below the decision-month benchmark for its position tier.
train["opportunity_label"] = (train["future_ctr"] < train["tier_benchmark_ctr"]).astype(int)
test["opportunity_label"] = (test["future_ctr"] < test["tier_benchmark_ctr"]).astype(int)

print("Training label base rate:", train["opportunity_label"].mean().round(3))
print("Test label base rate:    ", test["opportunity_label"].mean().round(3))
display(tier_benchmark.to_frame())


Training label base rate: 0.396
Test label base rate:     0.22


,tier_benchmark_ctr
position_tier,
deep,0.000000
page_1,0.186350
page_3_5,0.000000
striking,0.000000
top_3,0.206186


## 3. Train + compare vs my baseline

### Frozen Week-4 baseline

The hand-written rule is kept unchanged in spirit:

`positive CTR gap × log1p(impressions)`

where the CTR gap is current February CTR below the position-tier benchmark.

For this fair held-out comparison, the benchmark is the **training-only benchmark** defined above. No March information enters the score.

### Model

Logistic Regression uses the four clean decision-time features:
1. `feb_impressions`
2. `feb_clicks`
3. `feb_ctr`
4. `feb_avg_position`

`StandardScaler` is fit only on training data.


In [ ]:
FEATURES = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
]

# Keep the same frozen rule for the held-out comparison.
for frame in (train, test):
    frame["ctr_gap_pp"] = (
        frame["tier_benchmark_ctr"] - frame["feb_ctr"]
    ).clip(lower=0)

    frame["baseline_score"] = (
        frame["ctr_gap_pp"] * np.log1p(frame["feb_impressions"])
    )

# Model pipeline: preprocessing and fitting happen only on training rows.
model = Pipeline([
    ("scale", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )),
])

X_train = train[FEATURES]
y_train = train["opportunity_label"]

X_test = test[FEATURES]
y_test = test["opportunity_label"]

model.fit(X_train, y_train)

test["model_score"] = model.predict_proba(X_test)[:, 1]

print("Model trained.")


Model trained.


In [ ]:
# Primary and secondary ranking metrics.
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

def ranking_summary(y_true, scores):
    return {
        "Precision@10": precision_at_k(y_true, scores, 10),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Average precision": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
    }

baseline_metrics = ranking_summary(
    y_test,
    test["baseline_score"]
)

model_metrics = ranking_summary(
    y_test,
    test["model_score"]
)

comparison = pd.DataFrame(
    [baseline_metrics, model_metrics],
    index=["Week-4 baseline", "Logistic Regression"]
)

comparison["Base rate"] = y_test.mean()
display(comparison.round(3))

print(
    f"Baseline Precision@10: {baseline_metrics['Precision@10']:.3f}"
)
print(
    f"Model Precision@10:    {model_metrics['Precision@10']:.3f}"
)

lift = (
    model_metrics["Precision@10"] / baseline_metrics["Precision@10"]
    if baseline_metrics["Precision@10"] > 0
    else np.nan
)

print(
    f"Precision@10 lift vs baseline: "
    f"{lift:.2f}x" if np.isfinite(lift)
    else "Precision@10 lift unavailable because baseline Precision@10 is zero."
)


,Precision@10,Precision@50,Average precision,ROC-AUC,Base rate
Week-4 baseline,0.9,0.94,0.615,0.809,0.22
Logistic Regression,0.4,0.46,0.658,0.904,0.22


Baseline Precision@10: 0.900
Model Precision@10:    0.400
Precision@10 lift vs baseline: 0.44x


### Model interpretation

For Logistic Regression, the coefficient sign indicates the direction of association with the positive opportunity label **after standardization**.

These are associations in this held-out evaluation design, not causal effects.


In [ ]:
# Interpretable standardized coefficients.
coef = pd.Series(
    model.named_steps["logreg"].coef_[0],
    index=FEATURES
).sort_values(key=np.abs, ascending=False)

coef_table = coef.rename("standardized_coefficient").to_frame()
display(coef_table.round(4))


,standardized_coefficient
feb_clicks,-6.8944
feb_avg_position,-2.6137
feb_impressions,2.3063
feb_ctr,-0.5657


## 4. Errors and interpretation

The review below focuses on:
- the model's highest-ranked pages that are actually negative (false positives);
- positive pages the model ranked poorly (false negatives);
- whether the errors concentrate in low-volume or unusual-position cases.

This is more useful than claiming success from a single metric alone.


In [ ]:
# Error review: top model picks.
test_review = test[
    [
        "client_hash_id",
        "content_hash_id",
        "position_tier",
        "feb_impressions",
        "feb_ctr",
        "future_ctr",
        "opportunity_label",
        "baseline_score",
        "model_score",
    ]
].copy()

test_review["model_rank"] = (
    test_review["model_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

test_review["error_type"] = np.select(
    [
        (test_review["model_score"] >= test_review["model_score"].nlargest(10).min())
        & (test_review["opportunity_label"] == 0),
        (test_review["opportunity_label"] == 1)
        & (test_review["model_rank"] > 50),
    ],
    [
        "false_positive_in_top10",
        "positive_but_ranked_below_50",
    ],
    default="other"
)

top10 = (
    test_review
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("Top-10 model review:")
display(top10.round(4))

false_positives = (
    test_review[
        test_review["opportunity_label"].eq(0)
    ]
    .sort_values("model_score", ascending=False)
    .head(5)
)

false_negatives = (
    test_review[
        test_review["opportunity_label"].eq(1)
    ]
    .sort_values("model_score", ascending=True)
    .head(5)
)

print("\nHighest-ranked false positives:")
display(false_positives.round(4))

print("\nLowest-ranked positive cases:")
display(false_negatives.round(4))


Top-10 model review:


,client_hash_id,content_hash_id,position_tier,feb_impressions,feb_ctr,future_ctr,opportunity_label,baseline_score,model_score,model_rank,error_type
16340,client_23a62021009f63c4,content_2ac8c7995de53cd1,top_3,92128.0,0.0043,0.5192,0,2.3073,1.0,1,false_positive_in_top10
11035,client_23a62021009f63c4,content_44f34c0a90047651,top_3,90223.0,0.0144,0.0113,1,2.1882,1.0,2,other
17750,client_23a62021009f63c4,content_4fe94bdfd75c38f9,top_3,42351.0,0.0472,0.2060,1,1.6935,1.0,3,other
58292,client_fef1a8f436438636,content_ba462518dad435fc,page_3_5,69134.0,0.0564,0.0503,0,0.0000,1.0,4,false_positive_in_top10
49581,client_fef1a8f436438636,content_84a6bf3578312e90,striking,79986.0,0.0925,0.0930,0,0.0000,1.0,5,false_positive_in_top10
8305,client_23a62021009f63c4,content_c150891972f394a2,page_1,41585.0,0.0553,0.1462,1,1.3937,1.0,6,other
51237,client_fef1a8f436438636,content_fc0d3723ce5a9bba,page_1,38452.0,0.0416,0.0600,1,1.5280,1.0,7,other
3603,client_23a62021009f63c4,content_bdf60c86117079be,page_3_5,51346.0,0.0156,0.0107,0,0.0000,1.0,8,false_positive_in_top10
11807,client_23a62021009f63c4,content_2f09787bdf392b16,striking,34293.0,0.0000,0.0053,0,0.0000,1.0,9,false_positive_in_top10
11548,client_23a62021009f63c4,content_a3a1317f7c2bc3dd,page_3_5,47036.0,0.0276,0.0060,0,0.0000,1.0,10,false_positive_in_top10



Highest-ranked false positives:


,client_hash_id,content_hash_id,position_tier,feb_impressions,feb_ctr,future_ctr,opportunity_label,baseline_score,model_score,model_rank,error_type
16340,client_23a62021009f63c4,content_2ac8c7995de53cd1,top_3,92128.0,0.0043,0.5192,0,2.3073,1.0,1,false_positive_in_top10
58292,client_fef1a8f436438636,content_ba462518dad435fc,page_3_5,69134.0,0.0564,0.0503,0,0.0000,1.0,4,false_positive_in_top10
49581,client_fef1a8f436438636,content_84a6bf3578312e90,striking,79986.0,0.0925,0.0930,0,0.0000,1.0,5,false_positive_in_top10
3603,client_23a62021009f63c4,content_bdf60c86117079be,page_3_5,51346.0,0.0156,0.0107,0,0.0000,1.0,8,false_positive_in_top10
11807,client_23a62021009f63c4,content_2f09787bdf392b16,striking,34293.0,0.0000,0.0053,0,0.0000,1.0,9,false_positive_in_top10



Lowest-ranked positive cases:


,client_hash_id,content_hash_id,position_tier,feb_impressions,feb_ctr,future_ctr,opportunity_label,baseline_score,model_score,model_rank,error_type
16110,client_23a62021009f63c4,content_e7842b8465823f63,page_1,21414.0,0.9947,0.0235,1,0.0,0.0,18666,positive_but_ranked_below_50
12336,client_23a62021009f63c4,content_c131bffd37122fd1,page_1,11516.0,0.9031,0.1510,1,0.0,0.0,18547,positive_but_ranked_below_50
9191,client_23a62021009f63c4,content_d837bdbff756181f,page_1,5731.0,1.2214,0.1632,1,0.0,0.0,18368,positive_but_ranked_below_50
16412,client_23a62021009f63c4,content_b4e64ee7daac3871,page_1,5266.0,1.2723,0.0000,1,0.0,0.0,18318,positive_but_ranked_below_50
11265,client_23a62021009f63c4,content_e509d21cd53fbc77,page_1,463.0,4.7516,0.0000,1,0.0,0.0,18155,positive_but_ranked_below_50


In [ ]:
# Error slices by impression volume.
test["volume_bucket"] = pd.cut(
    test["feb_impressions"],
    bins=[99, 249, 499, 999, 4999, np.inf],
    labels=["100-249", "250-499", "500-999", "1k-4.9k", "5k+"],
)

error_by_volume = (
    test.groupby("volume_bucket", observed=True)
    .agg(
        n=("opportunity_label", "size"),
        positive_rate=("opportunity_label", "mean"),
        mean_model_score=("model_score", "mean"),
        model_precision=("opportunity_label", lambda s: np.nan),
    )
)

# Calculate precision of the top model-ranked 20% within each volume bucket.
rows = []
for bucket, part in test.groupby("volume_bucket", observed=True):
    if len(part) == 0:
        continue
    k = max(1, int(np.ceil(len(part) * 0.20)))
    p20 = precision_at_k(
        part["opportunity_label"].to_numpy(),
        part["model_score"].to_numpy(),
        k
    )
    rows.append({
        "volume_bucket": str(bucket),
        "n": len(part),
        "positive_rate": part["opportunity_label"].mean(),
        "top_20pct_precision": p20,
    })

error_by_volume = pd.DataFrame(rows)
display(error_by_volume.round(3))


,volume_bucket,n,positive_rate,top_20pct_precision
0,100-249,3131,0.292,0.708
1,250-499,2971,0.291,0.666
2,500-999,3379,0.275,0.670
3,1k-4.9k,6911,0.179,0.646
4,5k+,2304,0.072,0.315


### What the errors mean

The most important error pattern to inspect is whether the model's mistakes are concentrated among pages with only the minimum 100 impressions. CTR is inherently noisier at low volume, so a page can look like an opportunity simply because a small number of clicks moved its rate.

A second limitation is that this target is a **proxy outcome**: it says whether next-month CTR is below the February position-tier benchmark. It does not prove that refreshing a page will cause CTR to improve.

The model therefore supports prioritization and review, not causal claims.


In [ ]:
# Automatic claim summary: keeps the conclusion tied to the observed test metrics.
base_p10 = baseline_metrics["Precision@10"]
model_p10 = model_metrics["Precision@10"]

print("=== ML-08 conclusion ===")
if model_p10 > base_p10:
    print(
        f"On the held-out client test set, Logistic Regression improved "
        f"Precision@10 from {base_p10:.3f} to {model_p10:.3f}."
    )
else:
    print(
        f"On the held-out client test set, Logistic Regression did NOT improve "
        f"Precision@10 over the baseline ({base_p10:.3f} vs {model_p10:.3f})."
    )

print(
    "This is a predictive ranking result on the defined proxy label, "
    "not evidence of a causal refresh effect."
)


=== ML-08 conclusion ===
On the held-out client test set, Logistic Regression did NOT improve Precision@10 over the baseline (0.900 vs 0.400).
This is a predictive ranking result on the defined proxy label, not evidence of a causal refresh effect.


## 5. Self-check

Before committing this notebook:

- [x] Method choice is explained.
- [x] Split is grouped by client.
- [x] Test clients are unseen during model fitting.
- [x] Position-tier benchmark is derived from training clients only.
- [x] Model features are decision-time February fields.
- [x] March is used only for the outcome.
- [x] `trend_direction`, `trend_pct`, and other label-derived fields are excluded.
- [x] Baseline and model use the same held-out test rows and same Precision@K definition.
- [x] Model interpretation and error review are included.
- [x] The conclusion does not turn association into a causal claim.

### Deliverable

Save this executed notebook as:

`work/notebooks/w05_model.ipynb`

Then commit it to the `main` branch of the Flyrank repository and submit the repository URL.
